## Preprocessing data/Raw_data/reddit.l2.clean.500k

In [2]:
import pandas as pd

In [11]:
import os
os.getcwd()

os.path.exists(
    "../data/Raw_data/reddit.l2.clean.500K/reddit.Albania.txt.tok.clean.shf.500K.nometa.tc.noent.fw.url.lc"
)


True

In [ ]:
import glob
glob.glob("../data/**", recursive=True)


In [16]:
with open(
    "../data/Raw_data/reddit.l2.clean.500K/reddit.Albania.txt.tok.clean.shf.500K.nometa.tc.noent.fw.url.lc",
    encoding="utf-8"
) as f:
    lines = f.read().splitlines()

df = pd.DataFrame(lines, columns=["text"])
df.head()


,text
0,i know how big the club is . ''
1,so when we say that PERSON is paid MONEY /seas...
2,search about PERSON before opening your mouth .
3,the redhead at 0:35 i am PERCENT sure i 've se...
4,yes .


In [17]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 256716 entries, 0 to 256715
Data columns (total 1 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   text    256716 non-null  str  
dtypes: str(1)
memory usage: 2.0 MB


In [18]:
df["generated"] = 0

In [20]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 256716 entries, 0 to 256715
Data columns (total 2 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   text       256716 non-null  str  
 1   generated  256716 non-null  int64
dtypes: int64(1), str(1)
memory usage: 3.9 MB


In [21]:
df['word_count'] = df['text'].str.split().apply(len)

average_words = df['word_count'].mean()
print(f"Average number of words per text: {average_words:.2f}")

df['char_count'] = df['text'].str.len()

average_chars = df['char_count'].mean()
print(f"Average number of characters per text: {average_chars:.2f}")

Average number of words per text: 13.60
Average number of characters per text: 64.04


### For all files

In [22]:
# Defining Path 

input_folder = "../data/Raw_data/reddit.l2.clean.500K"
output_folder = "../data/Separated_csv_1/non_native_human"

In [23]:
os.makedirs(output_folder, exist_ok=True)

In [24]:
# List all .lc files in the folder
file_list = [f for f in os.listdir(input_folder) if f.endswith(".lc")]

In [25]:

# Function to check if row has actual text (not just URL or empty)
def is_valid_text(text):
    text = text.strip()
    if not text:
        return False
    # Remove rows that are only 'URL' tokens (or whitespace + URL)
    if all(token == "URL" for token in text.split()):
        return False
    return True

In [ ]:
# Process each file
for filename in file_list:
    file_path = os.path.join(input_folder, filename)
    
    # Read lines safely
    with open(file_path, encoding="utf-8") as f:
        lines = f.read().splitlines()
    
    # Create DataFrame
    df = pd.DataFrame(lines, columns=["text"])
    
    # Add 'generated' column = 0 (human)
    df["generated"] = 0
    
    # Remove rows with no substance
    df = df[df["text"].apply(is_valid_text)]
    
    # Strip spaces
    df["text"] = df["text"].str.strip()
    
    # Save as CSV
    output_file = os.path.join(output_folder, filename.replace(".lc", ".csv"))
    df.to_csv(output_file, index=False, encoding="utf-8")
    
    print(f"Processed and saved: {output_file}")

print("All files processed!")

Processed and saved: ../data/Separated_csv_1/non_native_human\reddit.Albania.txt.tok.clean.shf.500K.nometa.tc.noent.fw.url.csv
Processed and saved: ../data/Separated_csv_1/non_native_human\reddit.Armenia.txt.tok.clean.shf.500K.nometa.tc.noent.fw.url.csv
Processed and saved: ../data/Separated_csv_1/non_native_human\reddit.Australia.txt.tok.clean.shf.500K.nometa.tc.noent.fw.url.csv
Processed and saved: ../data/Separated_csv_1/non_native_human\reddit.Austria.txt.tok.clean.shf.500K.nometa.tc.noent.fw.url.csv
Processed and saved: ../data/Separated_csv_1/non_native_human\reddit.Belgium.txt.tok.clean.shf.500K.nometa.tc.noent.fw.url.csv
Processed and saved: ../data/Separated_csv_1/non_native_human\reddit.Bosnia.txt.tok.clean.shf.500K.nometa.tc.noent.fw.url.csv
Processed and saved: ../data/Separated_csv_1/non_native_human\reddit.Bulgaria.txt.tok.clean.shf.500K.nometa.tc.noent.fw.url.csv
Processed and saved: ../data/Separated_csv_1/non_native_human\reddit.Canada.txt.tok.clean.shf.500K.nometa.tc.

In [ ]:

# Folder containing processed human CSVs
input_folder = "../data/Separated_csv_1/non_native_human"

# Output file path
output_file = os.path.join(input_folder, "all_human_filtered.csv")

# List all CSV files
csv_files = [f for f in os.listdir(input_folder) if f.endswith(".csv")]

all_dfs = []

# Process each CSV
for file in csv_files:
    file_path = os.path.join(input_folder, file)
    
    # Load CSV
    df = pd.read_csv(file_path)
    
    # Drop rows where 'text' is NaN
    df = df.dropna(subset=['text'])
    
    # Ensure all entries in 'text' are strings
    df['text'] = df['text'].astype(str)
    
    # Count words per row
    df['word_count'] = df['text'].str.split().apply(len)
    
    # Keep only rows with 15 words or more
    df = df[df['word_count'] >= 15].copy()
    
    # Drop temporary column
    df.drop(columns=['word_count'], inplace=True)
    
    # Append to list
    all_dfs.append(df)

# Combine all filtered DataFrames
combined_df = pd.concat(all_dfs, ignore_index=True)

# Save combined CSV
combined_df.to_csv(output_file, index=False, encoding="utf-8")

print(f"All CSVs combined and saved at: {output_file}")
print(f"Total rows after filtering: {len(combined_df)}")


All CSVs combined and saved at: ../data/Separated_csv_1/non_native_human\all_human_filtered.csv
Total rows after filtering: 9095719
